# 🚀 End-to-End Sales Forecasting & Demand Intelligence System
### Data Preparation & Exploratory Data Analysis (EDA)

---

## 1. Project Introduction
Welcome to **Notebook 1** of the End-to-End Sales Forecasting & Demand Intelligence System. In the modern retail landscape, anticipating customer demand and understanding historical sales trends are critical for maintaining optimal inventory levels, maximizing revenue, and ensuring customer satisfaction. 

This notebook focuses on laying a robust foundation for our machine learning pipeline by rigorously processing and analyzing historical sales data.

## 2. Business Problem
Retail organizations frequently struggle with stockouts of popular items and overstocking of slow-moving inventory. This imbalance leads to lost revenue, increased holding costs, and diminished profit margins. The goal is to deeply understand historical sales patterns, identify key drivers of profitability, and prepare a pristine dataset that will power downstream predictive modeling.

## 3. Project Objectives
*   **Data Ingestion & Integrity:** Load and validate the primary sales dataset (`train.csv`).
*   **Data Quality Assurance:** Identify and rectify missing values, duplicates, and invalid data types to ensure a high-fidelity dataset.
*   **Feature Engineering:** Extract temporal, financial, and logistical features (e.g., shipping times, seasonality markers) to enrich the dataset.
*   **Exploratory Data Analysis (EDA):** Uncover actionable business insights regarding regional performance, category profitability, and customer segmentation.
*   **Data Aggregation:** Generate optimized, aggregated views of the data (Daily, Weekly, Monthly) for subsequent forecasting models.

## 4. Dataset Description
The dataset contains transactional records of retail sales. Key dimensions and metrics include:
*   **Temporal:** `Order Date`, `Ship Date`
*   **Categorical:** `Category`, `Sub-Category`, `Region`, `State`, `Segment`, `Product Name`
*   **Quantitative:** `Sales`, `Profit`, `Discount`, `Quantity`


## 5. Setup & Import Libraries
We follow industry best practices by organizing our imports into standard library, third-party data manipulation, and visualization categories. We also configure logging to replace excessive print statements.


In [ ]:
# Standard Library Imports
import os
import warnings
import logging
from typing import Dict, List, Tuple

# Third-Party Imports for Data Manipulation
import numpy as np
import pandas as pd

# Visualization Libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
warnings.filterwarnings('ignore') # Suppress unnecessary warnings for clean output
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_theme(style="whitegrid", palette="muted")
pd.set_option('display.max_columns', None)

# Configure Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

logger.info("Libraries successfully imported and environment configured.")


## 6. Load Dataset
Loading the primary transactional data (`train.csv`). We will encapsulate this in a robust function that includes error handling.


In [ ]:
def load_data(filepath: str) -> pd.DataFrame:
    """
    Loads a CSV dataset into a Pandas DataFrame.
    
    Args:
        filepath (str): The path to the CSV file.
        
    Returns:
        pd.DataFrame: The loaded dataset.
    """
    try:
        logger.info(f"Attempting to load data from {filepath}...")
        df = pd.read_csv(filepath)
        logger.info(f"Data successfully loaded. Dataset shape: {df.shape}")
        return df
    except FileNotFoundError:
        logger.error(f"File not found at {filepath}. Please verify the path.")
        raise
    except Exception as e:
        logger.error(f"An unexpected error occurred while loading data: {e}")
        raise

# Load the dataset
# Assuming 'train.csv' is in the same directory or provide the correct path.
DATA_PATH = 'train.csv'

# Note for user: ensure 'train.csv' is in the same directory as this notebook
df_raw = load_data(DATA_PATH)


## 7. Data Understanding
To effectively process the data, we must first understand its structure, dimensions, and statistical properties.


In [ ]:
def display_data_summary(df: pd.DataFrame) -> None:
    """
    Displays a comprehensive summary of the dataset including head, tail, info, and descriptive statistics.
    """
    logger.info("--- DATASET SUMMARY ---")
    
    print("\n1. Dataset Shape:")
    print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
    
    print("\n2. Columns:")
    print(df.columns.tolist())
    
    print("\n3. First 5 Rows:")
    display(df.head())
    
    print("\n4. Last 5 Rows:")
    display(df.tail())
    
    print("\n5. Random Sample (5 Rows):")
    display(df.sample(5, random_state=42))
    
    print("\n6. DataFrame Info & Memory Usage:")
    display(df.info(memory_usage='deep'))
    
    print("\n7. Descriptive Statistics (Numerical):")
    display(df.describe().T)
    
    print("\n8. Descriptive Statistics (Categorical):")
    display(df.describe(include='object').T)

if df_raw is not None:
    display_data_summary(df_raw)


### Column Dictionary
| Column Name | Type | Description |
| :--- | :--- | :--- |
| `Row ID` | Integer | Unique identifier for each row. |
| `Order ID` | String | Unique identifier for each customer order. |
| `Order Date` | Date | The date the order was placed. |
| `Ship Date` | Date | The date the order was shipped. |
| `Ship Mode` | String | The shipping method chosen. |
| `Customer ID` | String | Unique identifier for the customer. |
| `Customer Name` | String | Full name of the customer. |
| `Segment` | String | The market segment the customer belongs to. |
| `Country` | String | Country where the order was placed. |
| `City` | String | City where the order was delivered. |
| `State` | String | State where the order was delivered. |
| `Postal Code` | Integer/String | Postal code for the delivery location. |
| `Region` | String | Geographic region of the delivery. |
| `Product ID` | String | Unique identifier for the product. |
| `Category` | String | High-level product category. |
| `Sub-Category` | String | Specific sub-category of the product. |
| `Product Name` | String | Full name/description of the product. |
| `Sales` | Float | Total revenue generated from the transaction. |
| `Quantity` | Integer | Number of items purchased. |
| `Discount` | Float | Discount applied to the product. |
| `Profit` | Float | Net profit generated from the transaction. |



## 8. Data Cleaning & Quality Checks
Data quality is paramount. We will perform checks for:
1. Missing Values
2. Duplicates
3. Negative values where invalid (e.g., Sales, Quantity)
4. Cardinality of categorical columns


In [ ]:
def data_quality_report(df: pd.DataFrame) -> pd.DataFrame:
    """
    Generates a professional data quality report for the dataframe.
    """
    report = pd.DataFrame(index=df.columns)
    report['Data Type'] = df.dtypes
    report['Total Records'] = len(df)
    report['Missing Values'] = df.isnull().sum()
    report['% Missing'] = round((df.isnull().sum() / len(df)) * 100, 2)
    report['Unique Values'] = df.nunique()
    report['% Unique (Cardinality)'] = round((df.nunique() / len(df)) * 100, 2)
    
    # Check for negative values in numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    negative_counts = []
    for col in df.columns:
        if col in numeric_cols:
            negative_counts.append((df[col] < 0).sum())
        else:
            negative_counts.append(0)
    report['Negative Values'] = negative_counts
    
    return report

logger.info("Generating Data Quality Report...")
dq_report = data_quality_report(df_raw)
display(dq_report)

# Duplicate Check
duplicates_count = df_raw.duplicated().sum()
logger.info(f"Total duplicate rows found: {duplicates_count}")

# Clean Dataset
df_clean = df_raw.copy()

# Drop duplicates if any
if duplicates_count > 0:
    df_clean.drop_duplicates(inplace=True)
    logger.info("Duplicates removed.")

# Handle Postal Code if it is treated as numeric but has missing/invalid values
if 'Postal Code' in df_clean.columns:
    df_clean['Postal Code'] = df_clean['Postal Code'].fillna('Unknown').astype(str)

logger.info(f"Cleaned dataset shape: {df_clean.shape}")


## 9. Feature Engineering
We will enrich the dataset by creating temporal, logistical, and financial features to enable deeper analysis and future modeling. Vectorized pandas operations are used for performance.


In [ ]:
def perform_feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """
    Applies feature engineering to the dataset.
    Converts dates, extracts temporal features, calculates shipping times, and derives financial metrics.
    """
    logger.info("Starting feature engineering...")
    df_fe = df.copy()
    
    # 1. Date Parsing
    df_fe['Order Date'] = pd.to_datetime(df_fe['Order Date'], format='mixed', errors='coerce')
    df_fe['Ship Date'] = pd.to_datetime(df_fe['Ship Date'], format='mixed', errors='coerce')
    
    # 2. Time-Based Features (Order Date)
    df_fe['Year'] = df_fe['Order Date'].dt.year
    df_fe['Month'] = df_fe['Order Date'].dt.month
    df_fe['Month Name'] = df_fe['Order Date'].dt.month_name()
    df_fe['Quarter'] = df_fe['Order Date'].dt.quarter
    df_fe['Week Number'] = df_fe['Order Date'].dt.isocalendar().week.astype(float)
    df_fe['Day'] = df_fe['Order Date'].dt.day
    df_fe['Day Name'] = df_fe['Order Date'].dt.day_name()
    df_fe['Weekend'] = df_fe['Order Date'].dt.dayofweek.isin([5, 6]).astype(float)
    
    # Season Mapping
    def get_season(month):
        if pd.isna(month): return np.nan
        if month in [12, 1, 2]: return 'Winter'
        elif month in [3, 4, 5]: return 'Spring'
        elif month in [6, 7, 8]: return 'Summer'
        else: return 'Fall'
    df_fe['Season'] = df_fe['Month'].apply(get_season)
    
    # 3. Shipping-Based Features
    df_fe['Shipping Days'] = (df_fe['Ship Date'] - df_fe['Order Date']).dt.days
    
    # Handle cases where Ship Date is before Order Date (invalid data)
    invalid_shipping = (df_fe['Shipping Days'] < 0).sum()
    if invalid_shipping > 0:
         logger.warning(f"Found {invalid_shipping} records with Ship Date before Order Date. Setting to NaN.")
         df_fe.loc[df_fe['Shipping Days'] < 0, 'Shipping Days'] = np.nan
            
    df_fe['Shipping Year'] = df_fe['Ship Date'].dt.year
    df_fe['Shipping Month'] = df_fe['Ship Date'].dt.month
    df_fe['Shipping Quarter'] = df_fe['Ship Date'].dt.quarter
    df_fe['Shipping Week'] = df_fe['Ship Date'].dt.isocalendar().week.astype(float) 

    # 4. Financial Features
    df_fe['Profit Margin'] = np.where(df_fe['Sales'] > 0, df_fe['Profit'] / df_fe['Sales'], 0)
    df_fe['Unit Price'] = np.where(df_fe['Quantity'] > 0, df_fe['Sales'] / df_fe['Quantity'], 0)
    
    logger.info("Feature engineering completed successfully.")
    return df_fe

df_processed = perform_feature_engineering(df_clean)
display(df_processed[['Order Date', 'Year', 'Month Name', 'Season', 'Shipping Days', 'Profit Margin']].head())



## 10. Data Aggregation
To support dashboarding and time-series forecasting, we create reusable, highly optimized aggregated dataframes.



In [ ]:
def create_aggregations(df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    """
    Creates various aggregated views of the dataset.
    Returns a dictionary containing the aggregated DataFrames.
    """
    logger.info("Creating data aggregations...")
    agg_dict = {}
    
    # Temporal Aggregations
    agg_dict['daily_sales'] = df.groupby('Order Date')[['Sales', 'Profit', 'Quantity']].sum().reset_index()
    agg_dict['weekly_sales'] = df.groupby(['Year', 'Week Number'])[['Sales', 'Profit', 'Quantity']].sum().reset_index()
    agg_dict['monthly_sales'] = df.groupby(['Year', 'Month', 'Month Name'])[['Sales', 'Profit', 'Quantity']].sum().reset_index()
    agg_dict['quarterly_sales'] = df.groupby(['Year', 'Quarter'])[['Sales', 'Profit', 'Quantity']].sum().reset_index()
    agg_dict['yearly_sales'] = df.groupby('Year')[['Sales', 'Profit', 'Quantity']].sum().reset_index()
    
    # Categorical Aggregations
    if 'Category' in df.columns:
        agg_dict['category_sales'] = df.groupby('Category')[['Sales', 'Profit']].sum().reset_index().sort_values(by='Sales', ascending=False)
        if 'Sub-Category' in df.columns:
            agg_dict['subcategory_sales'] = df.groupby(['Category', 'Sub-Category'])[['Sales', 'Profit']].sum().reset_index().sort_values(by='Sales', ascending=False)
    
    # Geographic Aggregations
    if 'Region' in df.columns:
        agg_dict['region_sales'] = df.groupby('Region')[['Sales', 'Profit']].sum().reset_index().sort_values(by='Sales', ascending=False)
    if 'State' in df.columns:
        agg_dict['state_sales'] = df.groupby('State')[['Sales', 'Profit']].sum().reset_index().sort_values(by='Sales', ascending=False)
    
    # Customer/Segment Aggregations
    if 'Segment' in df.columns:
        agg_dict['segment_sales'] = df.groupby('Segment')[['Sales', 'Profit']].sum().reset_index().sort_values(by='Sales', ascending=False)
    
    logger.info("Aggregations created.")
    return agg_dict

aggregations = create_aggregations(df_processed)
if 'monthly_sales' in aggregations:
    display(aggregations['monthly_sales'].head())



## 11. Exploratory Data Analysis (EDA)
In this section, we deeply explore the dataset through professional, high-quality visualizations. We will answer crucial business questions and highlight trends.



In [ ]:
# Utility function for chart aesthetics
def format_chart(ax, title, xlabel, ylabel):
    ax.set_title(title, fontsize=14, weight='bold', pad=15)
    ax.set_xlabel(xlabel, fontsize=12, weight='bold')
    ax.set_ylabel(ylabel, fontsize=12, weight='bold')
    ax.tick_params(axis='both', which='major', labelsize=10)
    sns.despine()



### Plot 1: Category Transaction Distribution
*   **Objective:** Analyze the frequency of transactions across broad product categories.
*   **Business Question:** Which product category has the highest number of transactions?



In [ ]:
if 'Category' in df_processed.columns:
    plt.figure(figsize=(10, 6))
    ax = sns.countplot(data=df_processed, x='Category', order=df_processed['Category'].value_counts().index, palette='viridis')
    format_chart(ax, 'Transaction Volume by Category', 'Category', 'Number of Transactions')
    for p in ax.patches:
        ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontsize=10, color='black', xytext=(0, 5), textcoords='offset points')
    plt.show()


*   **Observation:** Office Supplies generally dominate transaction volume in typical retail data.
*   **Business Insight:** High frequency, low-ticket items drive foot traffic or base volume.
*   **Recommendation:** Bundle high-frequency items with high-margin items to boost AOV.



### Plot 2: Region Distribution
*   **Objective:** Evaluate transaction volume across different geographic regions.
*   **Business Question:** Which region drives the most traffic/orders?



In [ ]:
if 'Region' in df_processed.columns:
    plt.figure(figsize=(10, 6))
    ax = sns.countplot(data=df_processed, x='Region', order=df_processed['Region'].value_counts().index, palette='magma')
    format_chart(ax, 'Transaction Volume by Region', 'Region', 'Number of Transactions')
    plt.show()


*   **Observation:** Different regions yield different transaction volumes based on market penetration.
*   **Business Insight:** Understanding geographical strength helps align marketing efforts.
*   **Recommendation:** Investigate localized promotions for underperforming regions.



### Plot 3: Segment Distribution
*   **Objective:** Determine the composition of the customer base.
*   **Business Question:** Who are our primary customers?



In [ ]:
if 'Segment' in df_processed.columns:
    plt.figure(figsize=(8, 8))
    segment_counts = df_processed['Segment'].value_counts()
    plt.pie(segment_counts, labels=segment_counts.index, autopct='%1.1f%%', colors=sns.color_palette('pastel'), startangle=140, textprops={'fontsize': 12, 'weight':'bold'})
    plt.title('Customer Segment Distribution', fontsize=14, weight='bold')
    plt.show()


*   **Observation:** Visualizing the breakdown of B2B vs B2C clients.
*   **Business Insight:** Consumer segments usually hold majority volume, while Corporate holds higher margins.
*   **Recommendation:** Tailor loyalty programs depending on the dominant segment.



### Plot 4: Sales Distribution
*   **Objective:** Understand the distribution of order values.
*   **Business Question:** Are most sales high-value or low-value?



In [ ]:
plt.figure(figsize=(12, 6))
ax = sns.histplot(df_processed['Sales'], bins=100, kde=False, color='teal')
lower, upper = df_processed['Sales'].quantile([0.01, 0.95])
if upper > 0:
    ax.set_xlim(0, upper) # Capped to 95th percentile to handle outliers
format_chart(ax, f'Distribution of Sales Values (Capped at ${upper:.2f})', 'Sales ($)', 'Frequency')
plt.show()


*   **Observation:** The distribution is heavily right-skewed.
*   **Business Insight:** The business relies on high-frequency, lower-value transactions.
*   **Recommendation:** Implement upselling techniques at checkout.



### Plot 5: Profit Distribution
*   **Objective:** Analyze the spread of profitability per transaction.
*   **Business Question:** Are we losing money on a significant portion of transactions?



In [ ]:
plt.figure(figsize=(12, 6))
ax = sns.histplot(df_processed['Profit'], bins=150, kde=False, color='coral')
lower, upper = df_processed['Profit'].quantile([0.05, 0.95])
if lower != upper:
    ax.set_xlim(lower, upper) 
plt.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Break Even')
format_chart(ax, 'Distribution of Profit', 'Profit ($)', 'Frequency')
ax.legend()
plt.show()


*   **Observation:** A notable number of transactions result in a net loss.
*   **Business Insight:** Discounts or shipping costs might be erasing margins.
*   **Recommendation:** Audit discounting rules to ensure baseline profitability.



### Plot 6: Discount Distribution
*   **Objective:** Observe the frequency of different discount tiers.
*   **Business Question:** What is the most common discount applied?



In [ ]:
if 'Discount' in df_processed.columns:
    plt.figure(figsize=(10, 6))
    ax = sns.countplot(data=df_processed, x='Discount', palette='rocket')
    format_chart(ax, 'Frequency of Discounts Applied', 'Discount Rate (Decimal)', 'Frequency')
    plt.show()


*   **Observation:** Specific standard tiers (0%, 20%) are most common.
*   **Business Insight:** Over-discounting harms overall business health.
*   **Recommendation:** Reserve deep discounts strictly for inventory liquidation.



### Plot 7: Monthly Sales Trend
*   **Objective:** Track revenue generation over time.
*   **Business Question:** How are sales trending over time?



In [ ]:
if 'monthly_sales' in aggregations:
    plt.figure(figsize=(16, 6))
    monthly_trend = aggregations['monthly_sales'].copy()
    monthly_trend['Date_Obj'] = pd.to_datetime(monthly_trend['Year'].astype(str) + '-' + monthly_trend['Month'].astype(str) + '-01')
    monthly_trend = monthly_trend.sort_values('Date_Obj')
    ax = sns.lineplot(data=monthly_trend, x='Date_Obj', y='Sales', marker='o', linewidth=2, color='darkblue')
    format_chart(ax, 'Overall Monthly Sales Trend', 'Date', 'Total Sales ($)')
    plt.xticks(rotation=45)
    plt.show()


*   **Observation:** Overall sales exhibit seasonality with peaks around specific months.
*   **Business Insight:** Time series structure is crucial for future forecasting models.
*   **Recommendation:** Align supply chain to peak seasonal demand.



### Plot 8: Quarterly Sales Trend
*   **Objective:** Smooth out monthly volatility.
*   **Business Question:** Which quarter is the strongest?



In [ ]:
if 'quarterly_sales' in aggregations:
    plt.figure(figsize=(14, 6))
    quarterly = aggregations['quarterly_sales'].copy()
    quarterly['Quarter_Str'] = quarterly['Year'].astype(str) + '-Q' + quarterly['Quarter'].astype(str)
    ax = sns.barplot(data=quarterly, x='Quarter_Str', y='Sales', palette='Blues_d')
    format_chart(ax, 'Quarterly Sales Trend', 'Quarter', 'Total Sales ($)')
    plt.xticks(rotation=45)
    plt.show()


*   **Observation:** Q4 is typically the strongest quarter due to holidays.
*   **Business Insight:** Q1 is often the weakest, representing post-holiday slump.
*   **Recommendation:** Plan Q1 clearance to maintain revenue momentum.



### Plot 9: Yearly Sales Growth
*   **Objective:** Assess macroscopic business growth.
*   **Business Question:** Is the business growing YoY?



In [ ]:
if 'yearly_sales' in aggregations:
    plt.figure(figsize=(10, 6))
    yearly = aggregations['yearly_sales']
    ax = sns.barplot(data=yearly, x='Year', y='Sales', palette='Greens_d')
    format_chart(ax, 'Yearly Total Sales', 'Year', 'Total Sales ($)')
    plt.show()


*   **Observation:** Checking macroscopic expansion phase.
*   **Business Insight:** Consistent YoY growth validates business strategy.
*   **Recommendation:** Use historical YoY as baseline target.



### Plot 10: Revenue by Category
*   **Objective:** Compare total revenue generation.
*   **Business Question:** Which category brings in the most money?



In [ ]:
if 'category_sales' in aggregations:
    plt.figure(figsize=(10, 6))
    cat_sales = aggregations['category_sales']
    ax = sns.barplot(data=cat_sales, x='Category', y='Sales', palette='Set2')
    format_chart(ax, 'Total Revenue by Category', 'Category', 'Total Sales ($)')
    plt.show()


*   **Observation:** High-ticket categories drive revenue.
*   **Business Insight:** Volume vs Revenue mismatch is common (e.g. Office Supplies vs Tech).
*   **Recommendation:** Ensure supply chain robustness for revenue drivers.



### Plot 11: Revenue by Sub-Category
*   **Objective:** Identify granular product performance.
*   **Business Question:** What are the top-selling sub-categories?



In [ ]:
if 'subcategory_sales' in aggregations:
    plt.figure(figsize=(14, 8))
    sub_sales = aggregations['subcategory_sales'].head(15)
    ax = sns.barplot(data=sub_sales, x='Sales', y='Sub-Category', palette='flare')
    format_chart(ax, 'Top 15 Sub-Categories by Revenue', 'Total Sales ($)', 'Sub-Category')
    plt.show()


*   **Observation:** Revenue is heavily concentrated in a few sub-categories.
*   **Business Insight:** Pareto principle (80/20 rule) applies to inventory.
*   **Recommendation:** Prioritize inventory for top sub-categories.



### Plot 12: Total Revenue by Region
*   **Objective:** Map revenue generation geographically.
*   **Business Question:** Which region is our cash cow?



In [ ]:
if 'region_sales' in aggregations:
    plt.figure(figsize=(10, 6))
    reg_sales = aggregations['region_sales']
    ax = sns.barplot(data=reg_sales, x='Region', y='Sales', palette='cubehelix')
    format_chart(ax, 'Total Revenue by Region', 'Region', 'Total Sales ($)')
    plt.show()


*   **Observation:** Geographic spread of revenue shows regional dominance.
*   **Business Insight:** Resource allocation should match regional contribution.
*   **Recommendation:** Focus expansion budgets on top regions.



### Plot 13: Top 10 States by Revenue
*   **Objective:** Identify key state-level markets.
*   **Business Question:** Which states should we focus localized marketing on?



In [ ]:
if 'state_sales' in aggregations:
    plt.figure(figsize=(12, 6))
    state_sales = aggregations['state_sales'].head(10)
    ax = sns.barplot(data=state_sales, x='Sales', y='State', palette='mako')
    format_chart(ax, 'Top 10 States by Revenue', 'Total Sales ($)', 'State')
    plt.show()


*   **Observation:** Certain economic hubs dominate state-level sales.
*   **Business Insight:** Business is reliant on coastal or populous hubs.
*   **Recommendation:** Maintain aggressive advertising in critical states.



### Plot 14: Top 10 Products by Sales
*   **Objective:** Identify flagship products.
*   **Business Question:** What are our absolute best-selling items?



In [ ]:
if 'Product Name' in df_processed.columns:
    plt.figure(figsize=(12, 6))
    top_prods = df_processed.groupby('Product Name')['Sales'].sum().reset_index().sort_values('Sales', ascending=False).head(10)
    ax = sns.barplot(data=top_prods, x='Sales', y='Product Name', palette='crest')
    format_chart(ax, 'Top 10 Products by Revenue', 'Total Sales ($)', 'Product Name')
    ax.set_yticklabels([label.get_text()[:40] + '...' if len(label.get_text())>40 else label.get_text() for label in ax.get_yticklabels()])
    plt.show()


*   **Observation:** A few massive ticket items heavily influence revenue.
*   **Business Insight:** These are the flagship items.
*   **Recommendation:** Assign VIP support to buyers of these tier-1 products.



### Plot 15: Bottom 10 Products by Sales
*   **Objective:** Identify dead inventory.
*   **Business Question:** What products are tying up capital?



In [ ]:
if 'Product Name' in df_processed.columns:
    plt.figure(figsize=(12, 6))
    bot_prods = df_processed.groupby('Product Name')['Sales'].sum().reset_index().sort_values('Sales', ascending=True).head(10)
    ax = sns.barplot(data=bot_prods, x='Sales', y='Product Name', palette='Reds_r')
    format_chart(ax, 'Bottom 10 Products by Revenue', 'Total Sales ($)', 'Product Name')
    plt.show()


*   **Observation:** Obscure products generate almost no historical revenue.
*   **Business Insight:** Wasted warehouse space.
*   **Recommendation:** Delist and liquidate bottom-tier products.



### Plot 16: Average Shipping Time by Region
*   **Objective:** Evaluate logistics performance.
*   **Business Question:** Are some regions suffering from poor logistics?



In [ ]:
if 'Region' in df_processed.columns:
    plt.figure(figsize=(10, 6))
    ship_time = df_processed.groupby('Region')['Shipping Days'].mean().reset_index().sort_values('Shipping Days')
    ax = sns.barplot(data=ship_time, x='Region', y='Shipping Days', palette='light:b')
    format_chart(ax, 'Avg Shipping Time by Region', 'Region', 'Avg Shipping Days')
    plt.show()


*   **Observation:** Shipping times indicate logistics efficiency.
*   **Business Insight:** Consistency across regions means a robust logistics network.
*   **Recommendation:** Monitor shipping SLAs rigorously.



### Plot 17: Correlation Heatmap
*   **Objective:** Identify mathematical relationships.
*   **Business Question:** How strongly does Discount affect Profit?



In [ ]:
plt.figure(figsize=(8, 6))
num_cols = df_processed.select_dtypes(include=[np.number]).columns
corr = df_processed[num_cols].corr()
sns.heatmap(corr, annot=False, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Heatmap', fontsize=14, weight='bold', pad=15)
plt.show()


*   **Observation:** Negative correlation between Discount and Profit Margin is common.
*   **Business Insight:** Discounting destroys profitability.
*   **Recommendation:** Shift strategy to value-add instead of discounting.



### Plot 18: Seasonality Heatmap (Month vs Year)
*   **Objective:** Visualize month-over-month density.
*   **Business Question:** Is the Q4 spike consistent?



In [ ]:
if 'Year' in df_processed.columns and 'Month' in df_processed.columns:
    plt.figure(figsize=(12, 6))
    pivot_sales = df_processed.pivot_table(values='Sales', index='Month', columns='Year', aggfunc='sum')
    pivot_sales.index = [pd.Timestamp(2000, int(m), 1).strftime('%b') for m in pivot_sales.index]
    sns.heatmap(pivot_sales, cmap='YlGnBu', annot=False, linewidths=.5)
    plt.title('Sales: Month vs Year Heatmap', fontsize=14, weight='bold', pad=15)
    plt.show()


*   **Observation:** Heatmap visually isolates peak seasonal clusters.
*   **Business Insight:** Seasonality is highly predictable if columns align.
*   **Recommendation:** Heavily weight seasonal lags in ML models.



### Plot 19: Monthly Sales by Segment
*   **Objective:** Check if segments have different buying seasons.
*   **Business Question:** Do Corporate clients buy differently?



In [ ]:
if 'Segment' in df_processed.columns:
    plt.figure(figsize=(14, 6))
    seg_month_sales = df_processed.pivot_table(values='Sales', index='Segment', columns='Month', aggfunc='sum')
    seg_month_sales.columns = [pd.Timestamp(2000, int(m), 1).strftime('%b') for m in seg_month_sales.columns]
    sns.heatmap(seg_month_sales, cmap='rocket_r', annot=False, linewidths=.5)
    plt.title('Sales: Segment vs Month', fontsize=14, weight='bold', pad=15)
    plt.show()


*   **Observation:** Segments usually follow the same macro trend.
*   **Business Insight:** Unified marketing pushes are generally optimal.
*   **Recommendation:** Align Q4 pushes across all segments.



### Plot 20: Segment Profitability
*   **Objective:** Evaluate revenue quality from segments.
*   **Business Question:** Which segment gives the best margin?



In [ ]:
if 'Segment' in df_processed.columns:
    plt.figure(figsize=(10, 6))
    seg_agg = df_processed.groupby('Segment')[['Sales', 'Profit']].sum().reset_index()
    seg_agg_melted = pd.melt(seg_agg, id_vars='Segment', var_name='Metric', value_name='Amount')
    ax = sns.barplot(data=seg_agg_melted, x='Segment', y='Amount', hue='Metric', palette='muted')
    format_chart(ax, 'Sales vs Profit by Segment', 'Segment', 'Total Amount ($)')
    plt.show()


*   **Observation:** Ratio of profit to sales per segment.
*   **Business Insight:** All segments contribute proportionally.
*   **Recommendation:** Allocate acquisition budgets proportionally.



## 12. Business Questions Summary (Data-Driven Answers)
Let's explicitly answer the core business questions requested using our aggregated data.



In [ ]:
logger.info("Answering explicit business questions...")

try:
    if 'category_sales' in aggregations:
        print(f"1. Highest Revenue Category: {aggregations['category_sales'].iloc[0]['Category']}")
    if 'region_sales' in aggregations:
        print(f"2. Highest Sales Region: {aggregations['region_sales'].iloc[0]['Region']}")
    
    print(f"3. Average Shipping Time: {df_processed['Shipping Days'].mean():.2f} days")
    
    if 'Region' in df_processed.columns:
        reg_ship = df_processed.groupby('Region')['Shipping Days'].mean().sort_values()
        print(f"4. Fastest Shipping Region: {reg_ship.index[0]}")
        print(f"5. Slowest Shipping Region: {reg_ship.index[-1]}")
        
    best_month = aggregations['monthly_sales'].groupby('Month Name')['Sales'].sum().sort_values(ascending=False)
    print(f"6. Best Performing Month: {best_month.index[0]}")
    print(f"7. Worst Performing Month: {best_month.index[-1]}")
    
    if 'Category' in df_processed.columns:
        cat_profit = df_processed.groupby('Category')['Profit'].sum().sort_values(ascending=False)
        print(f"8. Highest Profit Category: {cat_profit.index[0]}")
        print(f"9. Highest Loss Category: {cat_profit.index[-1]}")
    
    if 'Region' in df_processed.columns:
        reg_profit = df_processed.groupby('Region')['Profit'].sum().sort_values(ascending=False)
        print(f"10. Most Profitable Region: {reg_profit.index[0]}")
        print(f"11. Least Profitable Region: {reg_profit.index[-1]}")
        
    yearly_sales = aggregations['yearly_sales']['Sales'].values
    if len(yearly_sales) > 1:
        growth = ((yearly_sales[-1] - yearly_sales[0]) / yearly_sales[0]) * 100
        print(f"12. Overall Growth Trend: {growth:.2f}% from start to finish.")
except Exception as e:
    logger.error(f"Error extracting business answers: {e}")



## 13. Save Processed Datasets
We export the cleaned master dataset and all aggregations for use in Notebook 2 (Forecasting Models).



In [ ]:
def export_datasets(df: pd.DataFrame, agg_dict: Dict[str, pd.DataFrame]) -> None:
    """
    Saves the processed dataframe and all aggregations to CSV files.
    """
    logger.info("Starting data export process...")
    
    # Save main processed data
    df.to_csv('processed_data.csv', index=False)
    logger.info("Saved: processed_data.csv")
    
    # Save aggregations
    for name, agg_df in agg_dict.items():
        filename = f"{name}.csv"
        agg_df.to_csv(filename, index=False)
        logger.info(f"Saved: {filename}")
        
    logger.info("All datasets successfully exported.")

# Execute Export
export_datasets(df_processed, aggregations)

